# PII Detection Baseline

Initial rule-based detection and redaction of synthetic Spanish-language applicant data.

**Scope:** email, phone, CURP-like and RFC-like identifiers.

**Limitation:** These rules are a prototype, not a validated PII detector.

In [7]:
import re
import pandas as pd

PII_PATTERNS = {
    'EMAIL': re.compile(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'
    ),
    'PHONE': re.compile(
        r'(?<!\w)(?:\+52[\s.-]?)?(?:\(?\d{2,3}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}(?!\w)'
    ),
    'CURP_LIKE': re.compile(
        r'\b[A-Z]{4}\d{6}[HM][A-Z]{5}[A-Z0-9]\d\b',
        re.IGNORECASE
    ),
    'RFC_LIKE': re.compile(
        r'\b[A-ZÑ&]{3,4}\d{6}[A-Z0-9]{3}\b',
        re.IGNORECASE
    ),
}

def detect_pii(text):
    findings = []
    for label, pattern in PII_PATTERNS.items():
        for match in pattern.finditer(text):
            findings.append({
                'label': label,
                'start': match.start(),
                'end': match.end(),
                'value': match.group(),
            })
    return sorted(findings, key=lambda item: item['start'])

def redact_pii(text):
    findings = detect_pii(text)
    for item in reversed(findings):
        replacement = f"[{item['label']}]"
        text = text[:item['start']] + replacement + text[item['end']:]
    return text

In [8]:
# Synthetic examples only; these are not real applicant records.
sample_text = """
Candidata: Ana Ejemplo
Correo: ana.ejemplo@example.com
Teléfono: +52 656 123 4567
CURP de prueba: GODE561231HDFXXX09
RFC de prueba: ABCD9001011A2
Experiencia: Analista de inventarios.
"""

findings = detect_pii(sample_text)
display(pd.DataFrame(findings))
print("\nRedacted document:\n")
print(redact_pii(sample_text))


,label,start,end,value
0,EMAIL,32,55,ana.ejemplo@example.com
1,PHONE,66,82,+52 656 123 4567
2,CURP_LIKE,99,117,GODE561231HDFXXX09
3,RFC_LIKE,133,146,ABCD9001011A2



Redacted document:


Candidata: Ana Ejemplo
Correo: [EMAIL]
Teléfono: [PHONE]
CURP de prueba: [CURP_LIKE]
RFC de prueba: [RFC_LIKE]
Experiencia: Analista de inventarios.

